# Simulação Analítica de Cavidade Elíptica para NV-Maser

Este notebook tem como objetivo modelar rigorosamente as propriedades eletromagnéticas de uma cavidade ressonante elíptica preenchida com material dielétrico de alta permissividade (como Rutilo ou Safira), focada em aplicações de NV-Maser a temperatura ambiente.

O objetivo principal é encontrar as condições geométricas e de contorno exatas para que ocorra a **degenerescência** entre os modos fundamentais $TE_{111}$ e $TM_{010}$ na frequência de transição do centro NV do diamante (2.87 GHz). 

Para garantir a precisão necessária para futuras etapas de fabricação de protótipos de alta complexidade em laboratórios de nanotecnologia, este modelo abandona aproximações cilíndricas e resolve a equação de Helmholtz diretamente no sistema de coordenadas elípticas através das **Funções de Mathieu**.

### Passo 1: Importação de Bibliotecas e Constantes

Iniciamos importando as ferramentas matemáticas de otimização e processamento numérico. O pacote `scipy.special` é fundamental aqui, pois fornece as rotinas para o cálculo das funções de Mathieu ($Ce_m$, $ce_m$, etc.).

Definimos também:
* $c_0 = 3 \times 10^{11}$ mm/s (Velocidade da luz no vácuo)
* Frequência alvo do centro NV = 2.87 GHz

In [31]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import matplotlib.patches as patches
import scipy.constants as sc
import scipy.special as sp
import scipy.optimize as opt
import ipywidgets as widgets
from IPython.display import display, clear_output

# ── constants ─────────────────────────────────────────────────
C0 = 3e11             # Velocidade da luz (mm/s)
TARGET = 2.87         # Frequência alvo em GHz
N = 200               # Resolução da Figuras

### Passo 2: Geometria Elíptica e Autovalores de Mathieu

Para uma cavidade com semi-eixo maior $a$ e excentricidade $e$, a distância focal é dada por $f = a \cdot e$. A fronteira metálica da cavidade é definida pela coordenada elíptica radial constante:
$$\xi_0 = \text{arccosh}\left(\frac{1}{e}\right)$$

A frequência de corte é governada pelo autovalor $k_c$, que compõe o parâmetro de Mathieu $q = (k_c f / 2)^2$. Os valores exatos de $k_c$ são encontrados zerando as funções na fronteira $\xi_0$:

* **Modo TM$_{010}$**: O campo elétrico longitudinal se anula na parede. Encontramos a primeira raiz de $Ce_0(\xi_0, q) = 0$.
* **Modo TE$_{111}$**: A derivada do campo magnético longitudinal se anula na parede. Encontramos a primeira raiz de $Ce'_1(\xi_0, q) = 0$.

A frequência final de ressonância do modo TE depende da altura da cavidade ($L$) e da permissividade do meio ($\varepsilon_r$):
$$f_{TE_{111}} = \sqrt{f_c^2 + f_z^2}$$
Onde $f_z = c_0 / (2L\sqrt{\varepsilon_r})$.

In [32]:
# ── physics (Mathieu Functions) ───────────────────────────────
def kc_mathieu(a, e, m, tipo):
    f = a * e
    xi_0 = np.arccosh(1.0 / e)
    
    def eq(kc):
        q = (kc * f / 2)**2
        val, der = sp.mathieu_modcem1(m, q, xi_0)
        return val if tipo == 'TM' else der

    # O guess é calculado de acordo com o valor de referencia do circular, para que o chute inicial seja mais proximo do valor real
    Reff = a * np.power(1 - e**2, 0.25)
    guess = (2.40483 if tipo == 'TM' else 1.84118) / Reff
    
    kcs = np.linspace(guess * 0.4, guess * 1.6, 120)
    valores = [eq(k) for k in kcs]
    mudancas = np.where(np.diff(np.sign(valores)))[0]
    
    if len(mudancas) > 0:
        idx = mudancas[0]
        try:
            return opt.brentq(eq, kcs[idx], kcs[idx+1])
        except:
            return guess
    return guess

def fTM(a, e, er):
    kc = kc_mathieu(a, e, 0, 'TM')
    return (C0 * kc / (2 * np.pi * np.sqrt(er))) / 1e9

def fTEcutoff(a, e, er):
    kc = kc_mathieu(a, e, 1, 'TE')
    return (C0 * kc / (2 * np.pi * np.sqrt(er))) / 1e9

def fTE(a, e, L, er):
    fc = fTEcutoff(a, e, er)
    fz = C0 / (2 * L * np.sqrt(er)) / 1e9
    return np.sqrt(fc**2 + fz**2)

def mode_volume(a, e, L):
    return np.pi * a * a * np.sqrt(1 - e**2) * L

### Passo 3: Busca pela Degenerescência e Volume Modal

Para o funcionamento do Maser, os modos $TE_{111}$ e $TM_{010}$ devem ser **degenerados**. Como a frequência do modo TM independe de $L$, utilizamos um método de **Bisseção** (Busca Binária) usando o otimizador do SciPy para varrer o espaço de soluções e encontrar o comprimento $L_{degen}$ exato onde as duas curvas de frequência se cruzam.

O volume do modo, um parâmetro importante para avaliar o confinamento do campo, é dado pela área da elipse multiplicada pela altura:
$$V = \pi a^2 \sqrt{1 - e^2} L$$

In [33]:
# ── fields (Mathieu Distribution) ─────────────────────────────
def compute_mathieu_field_unmasked(X, Y, a, e, m, tipo):
    f = a * e
    kc = kc_mathieu(a, e, m, tipo)
    q = (kc * f / 2)**2
    
    Z_complex = (X + 1j * Y) / f
    xi_eta = np.arccosh(Z_complex + 1e-15j)
    xi, eta = np.real(xi_eta), np.imag(xi_eta)
    
    v_cem = np.vectorize(lambda ang: sp.mathieu_cem(m, q, ang * 180.0 / np.pi)[0])
    v_mod = np.vectorize(lambda rad: sp.mathieu_modcem1(m, q, rad)[0])
    
    return np.real(v_mod(xi) * v_cem(eta))

def apply_ellipse_mask(X, Y, a, e, campo):
    mask = (X**2 / a**2) + (Y**2 / (a**2 * (1 - e**2))) <= 1.0
    return np.where(mask, campo, np.nan)

def F_Hz(X, Y, a, e):
    campo = compute_mathieu_field_unmasked(X, Y, a, e, 1, 'TE')
    return apply_ellipse_mask(X, Y, a, e, campo)

def F_Ez(X, Y, a, e):
    campo = compute_mathieu_field_unmasked(X, Y, a, e, 0, 'TM')
    return apply_ellipse_mask(X, Y, a, e, campo)

### Passo 4: Distribuição Espacial dos Campos Eletromagnéticos

A intensidade e a distribuição dos campos ditam o fator de preenchimento (filling factor) sobre o cristal de diamante. No sistema elíptico, a solução da equação de onda é o produto das funções de Mathieu radiais modificadas ($Ce_m$) e angulares normais ($ce_m$).

Mapeamos o plano complexo bidimensional $(x, y)$ para coordenadas elípticas $(\xi, \eta)$.
* Para o **TM$_{010}$**, modelamos o perfil do campo elétrico longitudinal $E_z$.
* Para o **TE$_{111}$**, modelamos o campo magnético longitudinal $H_z$ e extraímos as componentes transversais ($|H_\perp|$ e $|E_\perp|$) através de gradientes numéricos rigorosos sobre a malha espacial.

In [34]:
# ── optimization & UI helpers ─────────────────────────────────
def Ldegen_bisect(a, e, er, Lo=20.0, Hi=500.0, tol=1e-6):
    ftm = fTM(a, e, er)
    if fTE(a, e, Lo, er) <= ftm or fTE(a, e, Hi, er) >= ftm:
        return None
        
    for _ in range(80):
        mid = (Lo + Hi) / 2
        if fTE(a, e, mid, er) > ftm: Lo = mid
        else: Hi = mid
        if Hi - Lo < tol: break
    return {'L': (Lo + Hi) / 2, 'f': fTE(a, e, (Lo + Hi) / 2, er)}

def draw_geometry(ax, a, e):
    b = a * np.sqrt(1 - e**2)
    foc = a * e
    elipse = patches.Ellipse((0, 0), 2*a, 2*b, linewidth=2, edgecolor='darkcyan', facecolor='cyan', alpha=0.3)
    ax.add_patch(elipse)
    ax.plot([-foc, foc], [0, 0], marker='x', color='black', markersize=10, mew=2, linestyle='None', label="Focos")
    
    ax.set_title("Geometria da Seção Elíptica", fontsize=12)
    ax.set_xlim(-a*1.2, a*1.2)
    ax.set_ylim(-a*1.2, a*1.2)
    ax.set_aspect('equal')
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.set_xlabel("x (mm)", fontsize=10)
    ax.set_ylabel("y (mm)", fontsize=10)
    ax.legend(loc="upper right", fontsize=8)

def draw_field_with_peaks(ax, Ffn, a, e, title, cmap='inferno', N=150):
    b = a * np.sqrt(1 - e**2)
    foc = a * e
    x = np.linspace(-a * 1.05, a * 1.05, N)
    y = np.linspace(-b * 1.05, b * 1.05, N)
    X, Y = np.meshgrid(x, y)
    
    campo = Ffn(X, Y, a, e)
    campo_abs = np.abs(campo)
    
    # Normalização
    vm = np.nanmax(campo_abs) or 1.0
    Z_plot = campo_abs / vm
    
    niveis_cor = np.linspace(0, 1, 30)
    cf = ax.contourf(X, Y, Z_plot, levels=niveis_cor, cmap=cmap, vmin=0, vmax=1)
    ax.add_patch(patches.Ellipse((0, 0), 2*a, 2*b, color='black', fill=False, lw=2))
    
    # Focos
    ax.plot([-foc, foc], [0, 0], marker='x', color='cyan', markersize=10, mew=2, linestyle='None', label="Focos")
    
    # Detecção de picos com simetria espelhada
    idx_max = np.nanargmax(Z_plot)
    y_max_idx, x_max_idx = np.unravel_index(idx_max, Z_plot.shape)
    max_val = Z_plot[y_max_idx, x_max_idx]
    
    nx, ny = len(x), len(y)
    peaks_to_plot = set()
    for sx in [1, -1]:
        for sy in [1, -1]:
            sym_x_idx = x_max_idx if sx == 1 else (nx - 1) - x_max_idx
            sym_y_idx = y_max_idx if sy == 1 else (ny - 1) - y_max_idx
            if Z_plot[sym_y_idx, sym_x_idx] >= 0.99 * max_val:
                peaks_to_plot.add((X[sym_y_idx, sym_x_idx], Y[sym_y_idx, sym_x_idx]))
                
    x_p, y_p = X[y_max_idx, x_max_idx], Y[y_max_idx, x_max_idx]
    
    if abs(x_p) < 0.1 and abs(y_p) < 0.1:
        label_text = "Pico no centro"
    elif abs(y_p) < 0.1:
        label_text = f"Pico em x=±{abs(x_p):.1f} mm"
    elif abs(x_p) < 0.1:
        label_text = f"Pico em y=±{abs(y_p):.1f} mm"
    else:
        label_text = f"Pico (±{abs(x_p):.1f}, ±{abs(y_p):.1f}) mm"
        
    ax.plot([], [], marker='+', color='white', markeredgecolor='black', markersize=12, linestyle='None', label=label_text)
    
    for px, py in peaks_to_plot:
        ax.plot(px, py, marker='+', color='white', markeredgecolor='black', markersize=14, mew=1.5, linestyle='None')
        
    ax.legend(loc="upper right", fontsize=8, framealpha=0.9)
    plt.colorbar(cf, ax=ax)
    
    ax.set_title(title, fontsize=12)
    ax.set_xlim(-a*1.2, a*1.2)
    ax.set_ylim(-a*1.2, a*1.2)
    ax.set_aspect('equal')
    ax.set_xlabel("x (mm)", fontsize=10)
    
    # Remove ylabel if it's the 3rd plot to keep it clean (like your original code)
    if 'TE' in title:
        ax.set_ylabel("y (mm)", fontsize=10)

def draw_s11(ax, fte, ftm):
    flo = min(fte, ftm, TARGET) * 0.87
    fhi = max(fte, ftm, TARGET) * 1.13
    f   = np.linspace(flo, fhi, 3000)
    DEPTH = -35

    def S11_dip(fc, Q):
        d = 2 * Q * (f - fc) / fc
        return DEPTH / (1 + d**2)

    ax.axhline(0, color='grey', lw=0.8, alpha=0.5)
    ax.plot(f, S11_dip(fte, 9000), color='#1F77B4', lw=2.5, label=f'TE₁₁₁  {fte:.4f} GHz')
    ax.plot(f, S11_dip(ftm, 7000), color='#D62728', lw=2.5, label=f'TM₀₁₀  {ftm:.4f} GHz')
    ax.axvline(TARGET, color='#2CA02C', ls='--', lw=2, label=f'{TARGET} GHz')

    df = abs(fte - ftm) * 1000
    ax.annotate(f'|Δf| = {df:.3f} MHz', xy=(0.98, 0.08), xycoords='axes fraction',
                ha='right', va='bottom', fontsize=10, color='purple',
                bbox=dict(boxstyle='round,pad=0.3', fc='lightyellow', ec='purple', alpha=0.85))

    ax.set_ylim(DEPTH * 1.15, 5)
    ax.set_ylabel('S₁₁ (dB)', fontsize=10)
    ax.set_xlabel('Frequência (GHz)', fontsize=10)
    ax.set_title('S₁₁ Simulado (Reflexão)', fontsize=12)
    ax.legend(fontsize=9, loc='lower left')
    ax.grid(alpha=0.4)

def draw_Lscan(ax, a, e, er, Lcur):
    LSTART, Lmax = 1.0, max(40.0, Lcur * 2.5)
    Ls = np.linspace(LSTART, Lmax, 50)
    fTEs = [fTE(a, e, l, er) for l in Ls]
    ftmC = fTM(a, e, er)
    cross = Ldegen_bisect(a, e, er, Lo=LSTART, Hi=Lmax)

    ax.plot(Ls, fTEs, color='#1F77B4', lw=2.5, label='TE₁₁₁')
    ax.axhline(ftmC, color='#D62728', lw=2.5, label=f'TM₀₁₀ = {ftmC:.4f} GHz')
    ax.axhline(TARGET, color='#2CA02C', ls='--', lw=2, label='Alvo 2.87 GHz')

    if cross:
        Lc, fc = cross['L'], cross['f']
        ax.plot(Lc, fc, 'o', color='#27AE60', ms=12, label=f'L_degen = {Lc:.2f} mm')
        
        # Ajuste aqui: Lc - (Lmax-LSTART)*0.1 joga para a esquerda
        # fc + 0.25 joga para cima, saindo do meio da confusão de linhas
        # ha='right' alinha o final do texto com a seta
        ax.annotate(f'✓ {Lc:.2f} mm', xy=(Lc, fc), 
                    xytext=(Lc - (Lmax - LSTART)*0.10, fc + 0.25), 
                    ha='right',
                    arrowprops=dict(arrowstyle='->', lw=1.5), fontsize=10)

    ax.axvline(Lcur, color='orange', ls=':', lw=2.5, label=f'L atual = {Lcur:.1f} mm')
    ax.set_xlabel('Altura da Cavidade L (mm)', fontsize=10)
    ax.set_ylabel('Frequência (GHz)', fontsize=10)
    ax.set_title('Varredura de Degenerescência', fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.4)

### Passo 5: Geração dos Gráficos e Componentes de Análise

Nesta seção, definimos as sub-rotinas de desenho para compor o painel final. Para assegurar a fidelidade visual dos gradientes de intensidade, o grid numérico processa as séries de Mathieu em alta resolução.

* **Mapas 2D (Contourf)**: Renderização das seções transversais revelando os picos de intensidade e os focos da elipse.
* **Reflexão ($S_{11}$)**: Aproximação fenomênológica via mergulho Lorentziano para ilustrar a resposta de frequência esperada em uma medição VNA.
* **Varredura L-Scan**: Análise paramétrica que plota as curvas de ressonância e sinaliza graficamente o ponto exato da degenerescência.

In [35]:
# ── master figure ─────────────────────────────────────────────
def build_figure(a, e, L, er, N):
    plt.close('all')

    fte  = fTE(a, e, L, er)
    ftm  = fTM(a, e, er)
    df   = abs(fte - ftm) * 1000
    vm   = mode_volume(a, e, L) 
    
    onT  = abs(ftm - TARGET) / TARGET < 0.015
    degen = df < 20

    if degen and onT:
        st, sc = f'✓ DEGENERATE @ 2.87 GHz   |Δf| = {df:.3f} MHz', 'green'
    elif degen:
        st, sc = f'~ Degenerate mas fora do alvo   f(TM) = {ftm:.4f} GHz', 'orange'
    else:
        st, sc = f'✗ Not degenerate   |Δf| = {df:.3f} MHz', 'red'

    fig = plt.figure(figsize=(20, 13), dpi=120)
    fig.suptitle(
        f'a={a:.2f} mm   b={a*np.sqrt(1-e**2):.2f} mm   L={L:.2f} mm   '
        f'e={e:.3f}   ε_r={er:.2f}   Volume={vm/1000:.1f} cm³\n'
        f'f(TE)={fte:.4f} GHz   f(TM)={ftm:.4f} GHz\n{st}',
        fontsize=14, fontweight='bold', color=sc)

    # GridSpec 2x6:
    # Linha 0 (3 colunas): spans 0-2, 2-4, 4-6
    # Linha 1 (2 colunas): spans 0-3, 3-6
    gs = GridSpec(2, 6, figure=fig, hspace=0.35, wspace=0.6)

    # 1ª Linha: Geometria e Campos
    draw_geometry(fig.add_subplot(gs[0, 0:2]), a, e)
    draw_field_with_peaks(fig.add_subplot(gs[0, 2:4]), F_Hz, a, e, 'Modo eTE₁₁₁ ($|H_z|$)', N=N)
    draw_field_with_peaks(fig.add_subplot(gs[0, 4:6]), F_Ez, a, e, 'Modo eTM₀₁₀ ($|E_z|$)', N=N)

    # 2ª Linha: S11 e Altura da Cavidade
    draw_s11(fig.add_subplot(gs[1, 0:3]), fte, ftm)
    draw_Lscan(fig.add_subplot(gs[1, 3:6]), a, e, er, L)

    return fig

### Passo 6: Criação do Painel Principal e UI Interativa

O *Dashboard* final é estruturado através de um `GridSpec` customizado (2x6), organizando a visão espacial na linha superior e a resposta espectral/geométrica na linha inferior.

Utilizamos os controles dinâmicos do `ipywidgets` para manipulação paramétrica do semi-eixo maior ($a$), excentricidade ($e$), permissividade do meio ($\varepsilon_r$) e altura da cavidade ($L$). Para preservar a estabilidade computacional da modelagem exata, a renderização da imagem é engatilhada apenas ao soltar o *slider*.

In [36]:
# ── interface and execution ────────────────────────────────────
def run_interactive():
    style = {'description_width': '200px'}
    lay   = widgets.Layout(width='600px')

    sl = dict(
        er  = widgets.FloatSlider(value=80, min=70, max=90.0, step=0.05, description='ε_r  (1=ar, 80=Rutila):', style=style, layout=lay),
        a   = widgets.FloatSlider(value=4.91, min=1.0, max=10.0, step=0.01, description='a — semi-eixo maior (mm):', style=style, layout=lay),
        e   = widgets.FloatSlider(value=0.54, min=0.01, max=0.96, step=0.001, description='e — excentricidade:', style=style, layout=lay),
        L   = widgets.FloatSlider(value=8.20, min=1.0, max=20, step=0.01, description='L — altura (mm):', style=style, layout=lay),
    )
    
    for s in sl.values():
        s.continuous_update = False 

    out = widgets.Output()

    def update_plot(change=None):
        with out:
            clear_output(wait=True)
            fig = build_figure(sl['a'].value, sl['e'].value, sl['L'].value, sl['er'].value, N)
            display(fig)
            plt.close(fig)

    for s in sl.values():
        s.observe(update_plot, names='value')

    display(widgets.VBox([
        widgets.HTML('<h3>Dashboard NV-Maser</h3>'),
        *sl.values(),
        out
    ]))
    update_plot()

if __name__ == '__main__':
    run_interactive()